# F1 Strategy Predictor
Loads trained model and runs strategy simulations.
Run after 2_train_model.py has trained the model.

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from itertools import product

import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Disable GPU for inference (uses less memory, CPU is usually fine for inference)
# Comment this out if you want to use GPU
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

# --------------------
# Configuration
# --------------------
MODEL_FILE = "multitask_strategy_model.keras"
PREPROC_FILE = "multitask_preprocessing.joblib"

# Input files for simulation
RACE_CONTEXT_FILE = "race_context_input.csv"
FORECAST_FILE = "forecast_weather_detailed.csv"

WEATHER_FEATURES = ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]
VALID_COMPOUNDS = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]
DRY_COMPOUNDS = ["SOFT", "MEDIUM", "HARD"]
DEFAULT_PIT_LOSS = 22.0  # seconds

# Numeric feature order MUST match training
NUM_COLS = [
    "starting_position",
    "air_temp_mean", "air_temp_std",
    "track_temp_mean", "track_temp_std",
    "humidity_mean", "humidity_std",
    "rain_minutes_ratio", "rain_rate_mean", "rain_rate_std",
    "wind_speed_mean", "wind_speed_std",
    "temp_delta_mean", "rain_change_rate",
    "total_stint_laps", "avg_stint_laps", "max_stint_laps", "num_stints",
    "total_laps", "mean_tyre_life", "max_tyre_life",
    "mean_lap_time_sec", "lap_time_std_sec", "pace_trend_per_lap",
    "non_green_lap_ratio",
]

# --------------------
# Utility Functions
# --------------------
def safe_read_csv(path: Path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

def format_time(sec):
    m = int(sec // 60)
    s = sec - m * 60
    return f"{m}:{s:06.3f}"

def ensure_weather_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in WEATHER_FEATURES:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def compute_weather_seq_and_summary(weather_df: pd.DataFrame):
    if weather_df is None or len(weather_df) == 0:
        return None, None
    w = ensure_weather_columns(weather_df)
    w = w.dropna(subset=WEATHER_FEATURES, how="all")
    if w.empty:
        return None, None

    seq = w[WEATHER_FEATURES].ffill().bfill().fillna(0.0).values
    rain = w["rainfall"].fillna(0)
    temp_delta = w["track_temperature"] - w["air_temperature"]
    rain_change_rate = float(rain.diff().abs().mean()) if len(rain) > 1 else 0.0

    summary = {
        "air_temp_mean": float(w["air_temperature"].mean()),
        "air_temp_std": float(w["air_temperature"].std()),
        "track_temp_mean": float(w["track_temperature"].mean()),
        "track_temp_std": float(w["track_temperature"].std()),
        "humidity_mean": float(w["humidity"].mean()),
        "humidity_std": float(w["humidity"].std()),
        "rain_minutes_ratio": float((rain > 0).mean()),
        "rain_rate_mean": float(rain.mean()),
        "rain_rate_std": float(rain.std()),
        "wind_speed_mean": float(w["wind_speed"].mean()),
        "wind_speed_std": float(w["wind_speed"].std()),
        "temp_delta_mean": float(temp_delta.mean()),
        "rain_change_rate": float(rain_change_rate),
    }
    return seq, summary

def normalize_compound(name: str, default="MEDIUM"):
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return default
    s = str(name).upper().strip()
    if s not in VALID_COMPOUNDS:
        return default
    return s

# --------------------
# Load Model and Preprocessors
# --------------------
print("="*60)
print("F1 Strategy Predictor")
print("="*60)

assert Path(MODEL_FILE).exists(), f"Model not found: {MODEL_FILE}\nRun 2_train_model.py first!"
assert Path(PREPROC_FILE).exists(), f"Preprocessors not found: {PREPROC_FILE}\nRun 2_train_model.py first!"

print("\nLoading model and preprocessors...")
model = tf.keras.models.load_model(MODEL_FILE)
preproc = joblib.load(PREPROC_FILE)
print("✓ Model and preprocessors loaded")

# --------------------
# Input Encoding
# --------------------
def encode_inputs_for_model(team_name, track_name, starting_compound, starting_position, weather_seq, summary_overrides=None):
    """Encode inputs for model prediction."""
    
    def safe_transform(le, value, fallback_index=0):
        cls = list(le.classes_)
        if value in cls:
            return le.transform([value])[0]
        return fallback_index

    team_idx = safe_transform(preproc["team_le"], str(team_name))
    track_idx = safe_transform(preproc["track_le"], str(track_name))
    start_idx = safe_transform(preproc["start_le"], normalize_compound(starting_compound))

    # Summary defaults
    summaries = {k: 0.0 for k in NUM_COLS}
    summaries["starting_position"] = float(starting_position)

    if summary_overrides:
        for k, v in summary_overrides.items():
            if k in summaries:
                summaries[k] = float(v) if v is not None else 0.0

    x_num = np.array([summaries[c] for c in NUM_COLS], dtype="float32")
    x_num = (x_num - preproc["num_mean"]) / preproc["num_std"]

    # Weather sequence
    seq = np.asarray(weather_seq, dtype="float32")
    if seq.ndim != 2 or seq.shape[1] != len(WEATHER_FEATURES):
        raise ValueError(f"weather_seq must have shape [T, {len(WEATHER_FEATURES)}]")

    seq = pad_sequences([seq], maxlen=preproc["max_timesteps"], padding="post", dtype="float32")
    seq = (seq - preproc["seq_mean"]) / (preproc["seq_std"] + 1e-8)

    X = {
        "weather_seq": seq,
        "team_in": np.array([[team_idx]], dtype="int32"),
        "track_in": np.array([[track_idx]], dtype="int32"),
        "start_comp_in": np.array([[start_idx]], dtype="int32"),
        "num_in": np.array([x_num], dtype="float32"),
    }
    return X

def predict_race_outputs(X):
    """Make predictions using the model."""
    preds = model.predict(X, verbose=0)
    p_stops = preds["stops_out"][0]
    p_tire = preds["tire_out"][0]
    t_norm = float(preds["time_out"][0][0])

    # Denormalize time
    t_log = t_norm * preproc["y_time_std"] + preproc["y_time_mean"]
    t = float(np.expm1(t_log))

    return p_stops, p_tire, t

# --------------------
# Strategy Enumeration
# --------------------
def enumerate_strategies(max_stops=3):
    """Generate all possible tire strategy combinations."""
    strategies = []
    for stops in range(0, max_stops + 1):
        stints = stops + 1
        for comps in product(VALID_COMPOUNDS, repeat=stints):
            strategies.append({"stops": stops, "compounds": comps})
    return strategies

def strategy_score(base_race_time, pit_loss_pred, pit_stops, compounds, rain_ratio):
    """Score a strategy based on predicted time and penalties."""
    total = float(base_race_time) + float(pit_loss_pred) + float(pit_stops) * DEFAULT_PIT_LOSS

    # Penalties for inappropriate compound choices
    penalty = 0.0
    for c in compounds:
        c = normalize_compound(c)
        if rain_ratio > 0.2:
            # Rainy conditions: penalize dry tyres
            if c in DRY_COMPOUNDS:
                penalty += 15.0
        else:
            # Dry conditions: penalize wet/intermediate
            if c in ["INTERMEDIATE", "WET"]:
                penalty += 10.0

    total += penalty
    return total, penalty

# --------------------
# Strategy Simulation
# --------------------
def run_strategy_simulation(team_name, track_name, starting_position, starting_compound, 
                           forecast_weather_df, max_stops=3):
    """Run complete strategy simulation."""
    
    # Build weather sequence from forecast
    seq, ws = compute_weather_seq_and_summary(forecast_weather_df)
    if seq is None:
        raise ValueError("Forecast weather is empty / invalid")

    # Encode inputs and predict
    X = encode_inputs_for_model(
        team_name=team_name,
        track_name=track_name,
        starting_compound=starting_compound,
        starting_position=starting_position,
        weather_seq=seq,
        summary_overrides=ws,
    )
    p_stops, p_tire, pit_loss_pred = predict_race_outputs(X)

    # Get model's predicted stop count
    stops_pred = int(np.argmax(p_stops))
    try:
        stops_pred_label = int(preproc["stops_le"].inverse_transform([stops_pred])[0])
    except Exception:
        stops_pred_label = stops_pred

    # Baseline race time (relative ranking is what matters)
    base_race_time = 5400.0  # 90 minutes

    # Evaluate all strategies
    strategies = enumerate_strategies(max_stops=max_stops)
    out_rows = []
    rain_ratio = ws.get("rain_minutes_ratio", 0.0)

    for s in strategies:
        pit_stops = s["stops"]
        compounds = s["compounds"]
        
        total_time, penalty = strategy_score(base_race_time, pit_loss_pred, pit_stops, compounds, rain_ratio)
        
        out_rows.append({
            "pred_stops_argmax": stops_pred_label,
            "pit_loss_pred_sec": pit_loss_pred,
            "strategy_stops": pit_stops,
            "strategy_compounds": "-".join(compounds),
            "rain_ratio": rain_ratio,
            "total_time_sec": total_time,
            "penalty_sec": penalty,
        })

    out = pd.DataFrame(out_rows).sort_values("total_time_sec").reset_index(drop=True)
    out["total_time"] = out["total_time_sec"].apply(format_time)
    
    return out, p_stops, p_tire, ws

# --------------------
# Load Race Context
# --------------------
def load_race_context(path=RACE_CONTEXT_FILE):
    """Load race context from CSV or create default."""
    ctx = safe_read_csv(Path(path))
    if ctx.empty:
        print(f"Warning: {path} not found, using default values")
        ctx = pd.DataFrame([{
            "team_name": "Red Bull Racing",
            "track_name": "Monaco",
            "starting_position": 5.0,
            "starting_compound": "SOFT",
        }])
    return ctx

def load_forecast_weather(path=FORECAST_FILE):
    """Load forecast weather from CSV or create default."""
    w = safe_read_csv(Path(path))
    if w.empty:
        print(f"Warning: {path} not found, using default weather")
        w = pd.DataFrame([{
            "air_temperature": 25.0,
            "track_temperature": 35.0,
            "humidity": 60.0,
            "rainfall": 0.0,
            "wind_speed": 5.0,
        }])
    
    # Handle FastF1 column names
    w = w.copy()
    rename_map = {
        "AirTemp": "air_temperature",
        "TrackTemp": "track_temperature",
        "Humidity": "humidity",
        "Rainfall": "rainfall",
        "WindSpeed": "wind_speed",
    }
    for src, dst in rename_map.items():
        if src in w.columns and dst not in w.columns:
            w[dst] = w[src]
    
    w = ensure_weather_columns(w)
    return w

# --------------------
# Main Execution
# --------------------
if __name__ == "__main__":
    print("\n" + "="*60)
    print("Loading Race Context and Forecast")
    print("="*60)
    
    race_ctx = load_race_context()
    forecast_w = load_forecast_weather()
    
    print(f"\nRace context loaded: {len(race_ctx)} scenario(s)")
    print(f"Forecast weather: {len(forecast_w)} timestep(s)")
    
    # Run simulation for each row in race context
    for idx, row in race_ctx.iterrows():
        team_name = row.get("team_name", "Red Bull Racing")
        track_name = row.get("track_name", "Monaco")
        starting_position = float(row.get("starting_position", 10.0))
        starting_compound = normalize_compound(row.get("starting_compound", "MEDIUM"))
        
        print("\n" + "="*60)
        print(f"Running Strategy Simulation #{idx+1}")
        print("="*60)
        print(f"Team: {team_name}")
        print(f"Track: {track_name}")
        print(f"Starting Position: {starting_position}")
        print(f"Starting Compound: {starting_compound}")
        
        strategies_df, p_stops, p_tire, weather_summary = run_strategy_simulation(
            team_name=team_name,
            track_name=track_name,
            starting_position=starting_position,
            starting_compound=starting_compound,
            forecast_weather_df=forecast_w,
            max_stops=3,
        )
        
        print("\n--- Model Predictions ---")
        print("\nStop Count Distribution:")
        stops_classes = list(preproc["stops_le"].classes_)
        for i, cls in enumerate(stops_classes):
            print(f"  {cls} stops: {p_stops[i]*100:.1f}%")
        
        print("\nTire Compound Distribution:")
        tire_classes = list(preproc["tire_le"].classes_)
        for i, cls in enumerate(tire_classes):
            print(f"  {cls}: {p_tire[i]*100:.1f}%")
        
        print("\n--- Weather Summary ---")
        for key, val in weather_summary.items():
            print(f"  {key}: {val:.2f}")
        
        print("\n--- Top 10 Strategies (Ranked by Estimated Time) ---")
        print(strategies_df.head(10).to_string(index=False))
        
        # Save results
        output_file = f"strategy_output_{idx+1}.csv"
        strategies_df.to_csv(output_file, index=False)
        print(f"\n✓ Saved full results to: {output_file}")
    
    print("\n" + "="*60)
    print("Strategy Simulation Complete!")
    print("="*60)


2026-05-12 12:35:09.241305: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 12:35:09.255189: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-12 12:35:09.271369: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-12 12:35:09.276044: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 12:35:09.288642: I tensorflow/core/platform/cpu_feature_guar

F1 Strategy Predictor

Loading model and preprocessors...


2026-05-12 12:35:11.401656: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-05-12 12:35:11.401732: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:135] retrieving CUDA diagnostic information for host: jupyter-markvi2-ktu-lt---8679e81d
2026-05-12 12:35:11.401741: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:142] hostname: jupyter-markvi2-ktu-lt---8679e81d
2026-05-12 12:35:11.401905: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:166] libcuda reported version is: 565.57.1
2026-05-12 12:35:11.401937: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] kernel reported version is: 565.57.1
2026-05-12 12:35:11.401942: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:249] kernel version seems to match DSO: 565.57.1
/opt/conda/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:757: UserWarni

✓ Model and preprocessors loaded

Loading Race Context and Forecast

Race context loaded: 1 scenario(s)
Forecast weather: 14 timestep(s)

Running Strategy Simulation #1
Team: McLaren
Track: Imola
Starting Position: 4.0
Starting Compound: SOFT

--- Model Predictions ---

Stop Count Distribution:
  0 stops: 0.0%
  1 stops: 97.5%
  2 stops: 2.5%
  3 stops: 0.0%
  4 stops: 0.0%
  5 stops: 0.0%
  6 stops: 0.0%
  7 stops: 0.0%
  8 stops: 0.0%
  9 stops: 0.0%

Tire Compound Distribution:
  HARD: 87.7%
  HYPERSOFT: 0.0%
  INTERMEDIATE: 0.0%
  MEDIUM: 11.6%
  SOFT: 0.7%
  SUPERSOFT: 0.0%
  ULTRASOFT: 0.0%
  UNKNOWN: 0.0%

--- Weather Summary ---
  air_temp_mean: 23.79
  air_temp_std: 0.58
  track_temp_mean: 37.91
  track_temp_std: 2.15
  humidity_mean: 44.71
  humidity_std: 3.83
  rain_minutes_ratio: 0.00
  rain_rate_mean: 0.00
  rain_rate_std: 0.00
  wind_speed_mean: 2.84
  wind_speed_std: 0.41
  temp_delta_mean: 14.12
  rain_change_rate: 0.00

--- Top 10 Strategies (Ranked by Estimated Time) 